# Agente Mundial 2026 — LangGraph manual

Grafo con nodos explícitos: `agent` → `tools` → `agent` → ... → `END`

## 1. Instalación

In [9]:
%pip install -q langchain langchain-openai langgraph tavily-python requests


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Claves

In [ ]:
import os

os.environ["AZURE_OPENAI_ENDPOINT"]       = "tuendpoint"
os.environ["AZURE_OPENAI_API_KEY"]        = "tuapi"
os.environ["AZURE_OPENAI_DEPLOYMENT"]     = "gpt-4o-mini"      
os.environ["AZURE_OPENAI_API_VERSION"]    = "2024-02-15-preview"

os.environ["TAVILY_API_KEY"]              = "tuapi"

os.environ["EMAIL_FROM"]                  = "@gmail.com"
os.environ["EMAIL_PASSWORD"]              = "tucontraseñadeaplicacion"
os.environ["EMAIL_TO"]                    = "@gmail.com"

print("Claves configuradas.")

Claves configuradas.


## 3. Tools

In [11]:

import os
import smtplib
from datetime import date, timedelta
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from tavily import TavilyClient
from langchain.tools import tool


def _tavily_search(query: str, max_results: int = 5) -> str:
    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    resp = client.search(query=query, max_results=max_results, search_depth="advanced")
    results = resp.get("results", [])
    if not results:
        return "Sin resultados."
    return "\n".join(
        f"[{r['title']}]\n{r['content']}\nFuente: {r['url']}\n"
        for r in results
    )


@tool
def get_matches_today() -> str:
    """Busca los partidos del Mundial 2026 programados para hoy."""
    today = date.today().strftime("%B %d %Y")
    return _tavily_search(f"FIFA World Cup 2026 matches today {today} schedule kickoff time")


@tool
def get_next_matches() -> str:
    """Busca todos los partidos del Mundial 2026 de los próximos 3 días, día a día."""
    results = []
    for i in range(1, 4):
        day = date.today() + timedelta(days=i)
        text = _tavily_search(
            f"FIFA World Cup 2026 all matches {day.strftime('%B %d %Y')} complete schedule kickoff times",
            max_results=8
        )
        results.append(f"=== {day.isoformat()} ===\n{text}")
    return "\n\n".join(results)


@tool
def get_team_form(team_name: str) -> str:
    """Busca el estado de forma reciente de una selección. team_name en español o inglés."""
    return _tavily_search(
        f"{team_name} seleccion nacional forma reciente resultados Mundial 2026 jugadores clave",
        max_results=3
    )


@tool
def write_matches_txt(content: str, subject: str = "") -> str:
    """Escribe el análisis completo en partidos.txt. Si ya existe lo sobreescribe."""
    with open("partidos.txt", "w", encoding="utf-8") as f:
        f.write(content)
    subject_final = subject or f"Mundial 2026 — {date.today().strftime('%d/%m/%Y')}"
    with open("email_subject.txt", "w", encoding="utf-8") as f:
        f.write(subject_final)
    return f"partidos.txt escrito ({len(content)} chars). Asunto: '{subject_final}'."


@tool
def send_email_with_file(filepath: str = "partidos.txt") -> str:
    """Envía partidos.txt por email como cuerpo y adjunto. Llama siempre después de write_matches_txt."""
    from_addr = os.environ["EMAIL_FROM"]
    to_addr   = os.environ["EMAIL_TO"]
    password  = os.environ["EMAIL_PASSWORD"].replace(" ", "")

    try:
        subject = open("email_subject.txt", encoding="utf-8").read().strip()
    except FileNotFoundError:
        subject = f"Mundial 2026 — {date.today().strftime('%d/%m/%Y')}"

    try:
        body_text = open(filepath, encoding="utf-8").read()
    except FileNotFoundError:
        return f"ERROR: '{filepath}' no existe. Llama antes a write_matches_txt."

    if not body_text.strip():
        return "ERROR: partidos.txt vacío. Llama antes a write_matches_txt."

    msg = MIMEMultipart()
    msg["From"]    = from_addr
    msg["To"]      = to_addr
    msg["Subject"] = subject
    msg.attach(MIMEText(body_text, "plain", "utf-8"))
    with open(filepath, "rb") as f:
        part = MIMEBase("application", "octet-stream")
        part.set_payload(f.read())
    encoders.encode_base64(part)
    part.add_header("Content-Disposition", f'attachment; filename="{filepath}"')
    msg.attach(part)

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=10) as server:
            server.login(from_addr, password)
            server.sendmail(from_addr, to_addr, msg.as_string())
        return f"Email enviado a {to_addr}. Asunto: '{subject}'."
    except smtplib.SMTPAuthenticationError:
        return "ERROR auth: verifica verificación en 2 pasos y contraseña de aplicación."
    except Exception as e:
        return f"ERROR: {e}"


TOOLS = [get_matches_today, get_next_matches, get_team_form, write_matches_txt, send_email_with_file]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}
print("Tools:", list(TOOLS_BY_NAME.keys()))

Tools: ['get_matches_today', 'get_next_matches', 'get_team_form', 'write_matches_txt', 'send_email_with_file']


## 4. Grafo LangGraph manual

In [12]:
from typing import Annotated
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import SystemMessage, ToolMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict


# ── Estado del grafo ──────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


# ── LLM con tools vinculadas ──────────────────────────────────────
llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    temperature=0.7
)
llm_with_tools = llm.bind_tools(TOOLS)

SYSTEM_PROMPT = """Eres un analista de fútbol especializado en apuestas deportivas, con el estilo de Paolo Maldini: elegante, directo, conocedor del juego.

FLUJO OBLIGATORIO — ejecuta cada paso en orden:

PASO 1: Llama a get_matches_today.

PASO 2a — Si hay partidos hoy:
  - Para cada partido llama a get_team_form con el equipo local y con el visitante.
  - Redacta el texto completo con el FORMATO indicado.
  - Llama a write_matches_txt(content=<texto>, subject="Mundial 2026 - <Local> vs <Visitante> - <DD/MM/YYYY>").
  - Llama a send_email_with_file().

PASO 2b — Si NO hay partidos hoy:
  - Llama a get_next_matches.
  - Extrae TODOS los partidos de los 3 días sin omitir ninguno.
  - Para cada partido llama a get_team_form con ambos equipos.
  - Redacta el texto completo con el FORMATO indicado.
  - Llama a write_matches_txt(content=<texto>, subject="Mundial 2026 - Próximos partidos - <DD/MM/YYYY>").
  - Llama a send_email_with_file().

FORMATO (texto plano, sin markdown):

MUNDIAL 2026 — [DD/MM/YYYY]
================================================

[LOCAL] vs [VISITANTE]
Fecha: [DD/MM/YYYY a las HH:MM hora España CEST (UTC+2)] | [estadio, ciudad]

[2 líneas de contexto: grupo, qué se juegan]

Claves para apostar:
- 1X2: [quién gana — confianza alta/media/baja]
- Más/Menos 2.5 goles: [razonado]
- Ambos marcan: [sí/no, razonado]
- Handicap: [si hay favorito claro]
- APUESTA RECOMENDADA: [mejor valor esperado]

------------------------------------------------

REGLAS:
- Horas en hora española CEST (UTC+2), nunca UTC ni hora americana.
- Incluye TODOS los partidos, ordena del más cercano al más lejano.
- No escribas 'procederé a...'. Ejecuta las tools directamente.
- write_matches_txt y send_email_with_file son OBLIGATORIOS."""


# ── Nodo agente: llama al LLM ─────────────────────────────────────
def node_agent(state: AgentState) -> AgentState:
    print("[nodo: agent]")
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


# ── Nodo tools: ejecuta las tool calls del LLM ───────────────────
def node_tools(state: AgentState) -> AgentState:
    last_message = state["messages"][-1]
    tool_messages = []
    for tc in last_message.tool_calls:
        print(f"[nodo: tools] → {tc['name']}({list(tc['args'].keys())})")
        result = TOOLS_BY_NAME[tc["name"]].invoke(tc["args"])
        tool_messages.append(
            ToolMessage(content=str(result), tool_call_id=tc["id"])
        )
    return {"messages": tool_messages}


# ── Condición: seguir o terminar ──────────────────────────────────
def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


# ── Construcción del grafo ────────────────────────────────────────
graph_builder = StateGraph(AgentState)

graph_builder.add_node("agent", node_agent)
graph_builder.add_node("tools", node_tools)

graph_builder.set_entry_point("agent")

graph_builder.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", END: END}
)
graph_builder.add_edge("tools", "agent")

graph = graph_builder.compile()
print("Grafo compilado. Nodos:", list(graph_builder.nodes.keys()))

Grafo compilado. Nodos: ['agent', 'tools']


## 5. Ejecutar

In [13]:
from langchain_core.messages import HumanMessage

# thread_id identifica esta sesión — permite reanudar el grafo pausado
config = {"configurable": {"thread_id": "mundial_2026"}}

print("Fase 1: arrancando agente...\n")
graph.invoke(
    {
        "messages": [HumanMessage(content=(
            "Procesa los partidos del Mundial 2026 de hoy. "
            "Cuando termines escribe el análisis con write_matches_txt y PARA."
        ))],
        "approved": False
    },
    config=config
)

print("\nRevisa el análisis arriba")

Fase 1: arrancando agente...

[nodo: agent]
[nodo: tools] → get_matches_today([])
[nodo: agent]
[nodo: tools] → get_next_matches([])
[nodo: agent]
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: agent]
[nodo: tools] → write_matches_txt(['content', 'subject'])
[nodo: agent]
[nodo: tools] → send_email_with_file([])
[nodo: agent]

Revisa el análisis arriba


## 6. Human in the Loop — aprueba o rechaza el envío

Cambia `approved = True` para enviar el email, `False` para cancelar.

In [ ]:
approved = True

print(f"Decisión: {'ENVIAR' if approved else 'CANCELAR'}\n")

# Reanuda el grafo desde donde se pausó
graph.invoke(
    {"approved": approved},
    config=config
)

print("\nFlujo completado.")

Decisión: ENVIAR

[nodo: agent]
[nodo: tools] → get_matches_today([])
[nodo: agent]
[nodo: tools] → get_next_matches([])
[nodo: agent]
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: tools] → get_team_form(['team_name'])
[nodo: agent]
[nodo: tools] → write_matches_txt(['content', 'subject'])
[nodo: tools] → send_email_with_file([])
[nodo: agent]

Flujo completado.
